# Ginger Disease Classification — MobileNetV2 Transfer Learning

**Dataset:** `ginger_plant_dataset/` with two classes: `Bacterial_Wilt` and `Healthy`  
**Output:** `ginger_disease_model.tflite` — download and place in `streamlit_app/models/`

> **Colab instructions:**
> 1. Upload your dataset folder to Google Drive at `My Drive/Datasets/ginger_plant_dataset/`.
> 2. Run all cells in order (Runtime → Run all).
**Output:** `ginger_disease_model.tflite` — download and place in `streamlit_app/models/`


## Phase 1 â€” Environment & Drive Setup

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # â”€â”€ Dataset location in Google Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    DATASET_ROOT = '/content/drive/MyDrive/Datasets/ginger_plant_dataset'
else:
    DATASET_ROOT = os.path.join(os.path.dirname(os.getcwd()), 'ginger_plant_dataset')

TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
VAL_DIR   = os.path.join(DATASET_ROOT, 'validation')

HAS_VAL = os.path.exists(VAL_DIR)

print(f'DATASET_ROOT : {DATASET_ROOT}')
print(f'Train exists : {os.path.exists(TRAIN_DIR)}')
print(f'Val   exists : {HAS_VAL}  {"(will split from train)" if not HAS_VAL else ""}')

for split, d in [('train', TRAIN_DIR), ('validation', VAL_DIR)]:
    if not os.path.exists(d):
        continue
    classes = sorted([c for c in os.listdir(d) if os.path.isdir(os.path.join(d, c))])
    for cls in classes:
        n = len(os.listdir(os.path.join(d, cls)))
        print(f'  {split}/{cls}: {n} images')

## Phase 2 â€” Image Parameters & Data Generators

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 0.001
MODEL_SAVE = 'ginger_disease_model.h5'

print(f'Image size : {IMG_HEIGHT}x{IMG_WIDTH}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Max epochs : {EPOCHS}')

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

if HAS_VAL:
    # Separate train/ and validation/ folders exist
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest',
    )
    val_datagen = ImageDataGenerator(rescale=1.0 / 255)

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=True,
        seed=42,
    )
    val_gen = val_datagen.flow_from_directory(
        VAL_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=False,
    )

else:
    # No validation folder â€” use 80/20 split from train/
    print('No validation/ folder found. Using 20% of train data for validation.')
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest',
        validation_split=0.2,
    )

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='training',
        shuffle=True,
        seed=42,
    )
    val_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        subset='validation',
        shuffle=False,
        seed=42,
    )

import json
class_indices = train_gen.class_indices
print('class_indices:', class_indices)
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)
print('class_indices.json saved.')
print(f'Training samples  : {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')

## Phase 3 â€” Build MobileNetV2 Transfer Learning Model

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
)
base_model.trainable = False  # freeze pretrained weights

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)  # binary output

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## Phase 4 â€” Train with EarlyStopping & ModelCheckpoint

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        MODEL_SAVE,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
)

print(f'\nBest val_accuracy : {max(history.history["val_accuracy"]):.4f}')

## Phase 5 â€” Loss / Accuracy Plots

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history.history['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history.history['loss'],     label='Train Loss')
axes[0].plot(epochs_ran, history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_ran, history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(epochs_ran, history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

## Phase 6 â€” Confusion Matrix & Classification Report

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

val_gen.reset()
y_pred_prob = model.predict(val_gen, verbose=1)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_true = val_gen.classes

idx_to_class = {v: k for k, v in class_indices.items()}
class_names  = [idx_to_class[i] for i in sorted(idx_to_class)]

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Greens')
plt.title('Confusion Matrix â€” Validation Set')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

## Phase 7 â€” Sanity Test (one image from each class)

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image
import matplotlib.pyplot as plt

def predict_one(img_path, model, class_indices):
    img = keras_image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    arr = keras_image.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)
    prob = float(model.predict(arr, verbose=0)[0][0])  # P(Healthy)
    idx_to_class = {v: k for k, v in class_indices.items()}
    label = idx_to_class[1] if prob > 0.5 else idx_to_class[0]
    confidence = prob if prob > 0.5 else 1 - prob
    return label, confidence, img

for cls_name in sorted(class_indices.keys()):
    cls_dir = os.path.join(TRAIN_DIR, cls_name)
    sample  = os.listdir(cls_dir)[0]
    img_path = os.path.join(cls_dir, sample)
    label, conf, pil_img = predict_one(img_path, model, class_indices)
    print(f'True: {cls_name:20s}  â†’  Predicted: {label:20s}  (confidence {conf*100:.1f}%)')
    plt.figure(figsize=(3, 3))
    plt.imshow(pil_img)
    plt.title(f'True: {cls_name}\nPred: {label} ({conf*100:.1f}%)')
    plt.axis('off')
    plt.show()

## Phase 8 — Download Model (Colab only)

After training, place `ginger_disease_model.tflite` inside `streamlit_app/models/`.

In [ ]:
model_size_mb = os.path.getsize(MODEL_SAVE) / (1024 * 1024)
print(f'Model file: {MODEL_SAVE}  ({model_size_mb:.2f} MB)')
assert model_size_mb > 1, 'Model file is suspiciously small â€” check training output!'

if IN_COLAB:
    from google.colab import files
    files.download(MODEL_SAVE)
    files.download('class_indices.json')
    files.download('training_curves.png')
    files.download('confusion_matrix.png')
else:
    print(f'Local run â€” model saved to: {os.path.abspath(MODEL_SAVE)}')

## Phase 9 — Convert to TFLite & Download

Converts the trained Keras model to TFLite format, which runs on Python 3.13 via .  
Download  and place it in .

In [ ]:
import os
import numpy as np
import tensorflow as tf

H5_SAVE     = "ginger_disease_model.h5"
TFLITE_SAVE = "ginger_disease_model.tflite"
IMG_HEIGHT, IMG_WIDTH = 224, 224

# Load from .h5 if model is not already in memory (e.g. standalone / restarted kernel)
if "model" not in dir():
    print(f"Loading {H5_SAVE} from disk...")
    model = tf.keras.models.load_model(H5_SAVE)
    print("Model loaded.")

# Convert Keras model → TFLite (float32, no quantization)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()

with open(TFLITE_SAVE, "wb") as f:
    f.write(tflite_bytes)

tflite_mb = os.path.getsize(TFLITE_SAVE) / (1024 * 1024)
print(f"TFLite model: {TFLITE_SAVE}  ({tflite_mb:.2f} MB)")
assert tflite_mb > 0.5, "TFLite file suspiciously small — check model"

# Sanity check: run a dummy inference through the TFLite interpreter
tflite_interp = tf.lite.Interpreter(model_path=TFLITE_SAVE)
tflite_interp.allocate_tensors()
inp_det = tflite_interp.get_input_details()
out_det = tflite_interp.get_output_details()
dummy = np.zeros((1, IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.float32)
tflite_interp.set_tensor(inp_det[0]["index"], dummy)
tflite_interp.invoke()
tflite_out = float(tflite_interp.get_tensor(out_det[0]["index"])[0][0])
print(f"TFLite sanity check passed — dummy P(Healthy): {tflite_out:.4f}")

if "IN_COLAB" in dir() and IN_COLAB:
    from google.colab import files
    files.download(TFLITE_SAVE)
    print(f"Downloading {TFLITE_SAVE} — place in streamlit_app/models/ folder")
else:
    print(f"Local run — TFLite model saved to: {os.path.abspath(TFLITE_SAVE)}")